> 所谓xx并行，就是xx拆到不同的卡上，
- 数据并行，就是数据拆到不同的卡上（不同卡处理不同的数据）；
- 张量并行，就是 tensor 拆分到不同的卡上；

In [2]:
from IPython.display import Image

- MoE 训练用 TP 还是 EP？
    - 总参数与激活量参数计算
- https://huggingface.co/blog/moe
- https://arxiv.org/pdf/2401.06066
    - DeepSeekMoE
        - Experts: routed expert & shared expert

In [3]:
Image(url='./figs/deepseek-moe.png', width=400)

- 2/N => 4/2N => (1+3)/2N

## MoE

> MoE 是对 transformer feed-forward network 的修改
- MoE transformer block
    - shared part：自注意力层；
        - 这些是所有Token都必须经过的层。在Transformer中，这主要是自注意力（Self-Attention）层和一些层归一化（LayerNorm）等。这些部分的参数量相对较小。
        - 对于共享部分（如Attention层），我们采用经典的数据并行（DP）。这意味着，Attention层的完整参数被复制到了每一个GPU上。
    - moe part：gating network/router + experts
        - 这特指MoE层中的多个专家网络（Experts）。这部分的参数量是整个模型中最大的。一个专家本身就是一个FFN（前馈网络）。
        - 对于专家部分（Experts），我们采用专家并行（EP）。这意味着，所有专家的总参数被分散（sharded）到了不同的GPU上，每个GPU只持有总专家库的一个子集。

### 参数量计算

- DeepseekV3DecoderLayer
    - 每个解码器层都遵循标准的 Pre-LN（层归一化前置）结构，包含两个主要子模块：
        - 自注意力模块 (DeepseekV3Attention)：负责捕捉序列中不同 token 之间的依赖关系。
        - 前馈网络 (MLP 或 MoE)：负责对每个 token 的表示进行非线性变换。
    - 每个子模块都包裹在残差连接（Residual Connection）中，以保证训练的稳定性和信息的有效流动 。
- DeepseekV3Attention
    - DeepSeek-V3 没有使用标准的多头注意力（MHA）或分组查询注意力（GQA），而是采用了一种类似 LoRA 的瓶颈（bottleneck）结构来生成 Q、K、V 矩阵，这有助于减少参数量 。
    - Q/KV 投影：隐藏状态首先通过一个较窄的线性层（例如 q_a_proj）压缩，经过 RMSNorm 归一化后，再通过另一个线性层（q_b_proj）扩展到最终的目标维度。
    - 部分旋转位置编码 (Partial RoPE)：一个显著的特点是，Query 和 Key 的头部向量被分为了两部分。只有一部分维度（qk_rope_head_dim = 64）会应用旋转位置编码（RoPE），而另一部分（qk_nope_head_dim = 128）则不应用。这可能是一种旨在平衡绝对位置信息和相对位置信息的策略。
    - Yarn RoPE 缩放：模型使用了 "Yarn" (Yet another RoPE extension) 缩放方法，通过复杂的插值和修正，将模型的上下文窗口从预训练的 4096 token 有效扩展到 163840 token，同时保持了良好的性能 。
- DeepseekV3MoE
    - 这是模型参数效率的核心。在 61 个解码器层中，前 3 层使用标准的稠密 MLP，而后 58 层则使用 MoE 结构 。
        - 门控网络 (MoEGate)：对于每个 token，门控网络会计算一个分数，并根据这个分数从 256 个 "路由专家" 中选择最相关的 8 个（num_experts_per_tok = 8）。
        - 专家网络：
            - 路由专家 (Routed Experts)：模型拥有 256 个独立的、较小的 MLP 网络。在推理时，只有被门控网络选中的 8 个专家会被激活并参与计算。
            - 共享专家 (Shared Expert)：除了路由专家，还有一个所有 token 都会经过的共享专家 MLP。
        - 输出整合：最终的输出是 8 个被激活的路由专家输出的加权和，再加上共享专家的输出。这种设计结合了专业化（路由专家）和通用化（共享专家）的知识。
            - 直接加和

> deepseek v3 & r1 architecture

- 词汇表大小 (Vocabulary Size): $V = 129280$
- 隐藏层维度 (Hidden Size): $H = 7168$ (7*1024)
- 注意力头数 (Attention Heads): $N_h = 128$
- Q LoRA 秩 (Q LoRA Rank): $R_q=1536$
- KV LoRA 秩 (KV LoRA Rank): $R_{kv}=512$
- RoPE 维度 (QK RoPE Head Dim): $D_{rope}=64$
- 非 RoPE 维度 (QK NoPE Head Dim): $D_{nope} = 128$
- Q 头部总维度 (Q Head Dim): $D_q = D_{rope} + D_{nope} = 64 + 128 = 192$
- V 维度 (V Head Dim): $D_v = 128$
- 稠密层中间维度 (Dense MLP Intermediate Size): $I_{dense} = 18432$
- MoE 专家中间维度 (MoE Expert Intermediate Size): $I_{moe} = 2048$
- 路由专家数量 (Routed Experts): $E_r=256$
- 共享专家数量 (Shared Experts): $E_s = 1$
- 每 Token 激活专家数 (Experts per Token): $k=8$
- 前置稠密层数 (First k Dense Layers): $L_{dense} = 3$
- MoE 层数 (Number of MoE Layers): $L_{moe} = L - L_{dense} = 61 - 3 = 58$

https://zhuanlan.zhihu.com/p/21455638257
- 单个稠密 FFN 总参数(DeepseekV3MLP):
    - $P_{ffn\_dense\_single} = P_{gate} + P_{up} + P_{down} = 3 \times (H \times I_{dense}) = 3 \times 132,120,576 = 396,361,728$

### EP

- 2个gpu `[GPU0, GPU1]`，4个experts `[E0, E1, E2, E3]`，6个input tokens `[T0, T1, T2, T3, T4, T5]`
- DP & EP
    - GPU-0 存储着:一份完整的Attention层参数（副本），一份完整的Router参数（副本，因为它也算共享部分），专家 E0 和 E1 的参数（独有）
    - GPU-1 存储着:一份完整的Attention层参数（副本），一份完整的Router参数（副本），专家 E2 和 E3 的参数（独有）
- 第一步： 计算Attention层 (纯数据并行)
    - GPU-0 使用它本地的Attention层副本，独立计算出 `[T1, T2, T3]` 的Attention输出。
    - GPU-1 使用它本地的Attention层副本，独立计算出 `[T4, T5, T6]` 的Attention输出。
    - 到此为止，一切都和标准的数据并行完全一样。
- 第 2 步: 进入MoE层 (从DP切换到EP模式)
    - 路由计算:
        - GPU-0 上的Router副本计算出 `[T1, T2, T3]` 的目标专家。
            - GPU-0 持有 `[T1, T2, T3]`，并且知道它们的目的地分别是 `[E2, E0, E2]`。
        - GPU-1 上的Router副本计算出 `[T4, T5, T6]` 的目标专家。
            - GPU-1 持有 `[T4, T5, T6]`，并且知道它们的目的地分别是 `[E3, E0, E1]`。
    - All-to-All 通信:
        - Token们根据目标专家的位置，被发送到对应的GPU。
        - 例如，T1（在GPU-0上）需要去E2（在GPU-1上），所以T1的向量数据通过网络被发送到GPU-1。
            - GPU-0 收到了 来自 GPU-1 的 `[T5, T6]`。它现在持有的Token是 `[T2, T5, T6]`。
            - GPU-1 收到了 来自 GPU-0 的 `[T1, T3]`。它现在持有的Token是 `[T4, T1, T3]`。
        - Token不再按照原始批次分组，而是按照它们需要访问的专家所在的位置重新分组了。
    - 专家计算:
        - GPU-0 使用它独有的专家E0和E1进行计算。
        - GPU-1 使用它独有的专家E2和E3进行计算。
    - 第二次 All-to-All 通信:
        - 计算结果被送回Token原始的GPU。
    - 聚合:每个GPU将收到的结果聚合，完成MoE层的计算。